In [ ]:
# ClashBot autopilot -- the whole bot in one cell.
#
# One run = CYCLES play cycles. Each cycle:
#   1. recover  -- clear the connection-lost dialog, verify the home village
#   2. collect  -- tap every verified gold/elixir bubble
#   3. plan     -- read base state (builders, research, dialogs) and decide
#   4. upgrade  -- one upgrade scan when a free builder is verified
#   5. attack   -- find a match and deploy a verified loot wave (per ATTACK_MODE)
# Every cycle is appended to captures/play/<SESSION>/play_log.jsonl with
# screenshots for each attack, so a session is fully replayable.

import os, sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src" / "clashbot").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)  # asset catalogs and template paths are repo-relative

from clashbot import emulators
from clashbot.adb_client import AdbClient
from clashbot.player import GamePlayer

# ---- settings -------------------------------------------------------------
SERIAL      = None       # None auto-detects the first running emulator
CYCLES      = 5          # play cycles in this run
INTERVAL    = 90.0       # seconds between cycles
DRY_RUN     = True       # True = recognize/plan/report only, never tap. Flip to False to play.
ATTACK_MODE = "policy"   # "off" | "policy" (attack only when base work is clear) | "always"
SESSION     = "notebook_autoplay"
# ----------------------------------------------------------------------------

if SERIAL is None:
    devices = [d for d in emulators.discover() if d.state == "device"]
    if not devices:
        raise RuntimeError("no emulator found - start MEmu and enable ADB debugging")
    SERIAL = devices[0].serial
print(f"playing on {SERIAL} (dry_run={DRY_RUN}, attack={ATTACK_MODE}, cycles={CYCLES})")

report = GamePlayer(AdbClient(SERIAL)).run(
    session=SESSION, cycles=CYCLES, interval=INTERVAL,
    dry_run=DRY_RUN, attack_mode=ATTACK_MODE,
)

print(f"\nsession log -> {report.log_path}")
for cycle in report.cycles:
    print(f"cycle {cycle.cycle}: home={cycle.home} collected={cycle.collected} "
          f"buildings={cycle.buildings_seen} "
          f"layout={cycle.layout_stable}/{cycle.layout_total} stable "
          f"plan={cycle.plan_action} attacked={cycle.attack_attempted} "
          f"troops={len(cycle.troops_deployed)}")
    if cycle.building_counts:
        print("  buildings seen: " + ", ".join(
            f"{name} x{count}" for name, count in cycle.building_counts.items()))
    if cycle.town_hall_seen:
        print("  town hall: " + (f"level {cycle.town_hall_level}"
                                 if cycle.town_hall_level else "seen (level unknown)"))
    known = {name: value for name, value in cycle.resources.items() if value is not None}
    if known:
        print("  resources: " + ", ".join(f"{name}={value:,}" for name, value in known.items()))
    print(f"  plan reason: {cycle.plan_reason}")
    for note in cycle.notes:
        print(f"  - {note}")

# Final layout summary: what the bot believes the base contains so far.
from clashbot.layout import BaseLayout
layout = BaseLayout(Path(report.log_path).parent / "layout.json")
stable = layout.counts(stable_only=True)
tentative = {name: count - stable.get(name, 0)
             for name, count in layout.counts().items()
             if count - stable.get(name, 0) > 0}
print("\nbase layout so far:")
for name, count in stable.items():
    print(f"  {name} x{count} (confirmed)")
for name, count in tentative.items():
    print(f"  {name} x{count} (tentative - seen once)")
